# Goal and functionality of algorithm
Initiate organisms and watch them respond to stimuli through time. After a fixed number of time-steps, a reproduction condition triggers. Upon reproduction, the organisms which meet reproduction condition(s) reproduce (with a chance for mutation), all of the previous generation dies. This sequence repeats indefinitely. 


# Steps for a single generation
1. Initiate world (including state of n organisms) at t = 0
2. Allow organisms to perceive their situation and make a decision as to what to do. Taking action if applicable
3. Repeat step 2 for all timesteps

# Inter-generational steps
1. Run a single generation
2. Evaluate and execute reproduction condition. Reproduction should enable both passing of genetic information as well as the addition of new genetic information through mutation.
3. Repeat step 2 for n generations

# What does the MVP look like?
- organisms have a small number of neurons that map to some perception/action workflow
    - does this always need to look like [perception] -> [action]
- organisms can update state based on some perception of the world
- a population of organisms can reproduce based on some condition
- organisms can evolve (including passing genes and random mutations)

---
# Scratchpad

The MVP above is built. `python execute.py` runs it with the animation; the cells below are for poking at individual creatures and working out *why* a behaviour appeared.

The answer to "does this always need to look like [perception] -> [action]" turned out to be no: inner neurons keep their value between timesteps, so a genome can wire perception -> memory -> action, or a loop that ignores perception entirely.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

import brain_utils
import capability_utils
import settings
from organism import CRITERIA, organism, world

## One organism, up close

In [ ]:
sim = world(n_organisms=250)
org = sim.organism_states[0]

# Everything it can sense right now.
org.perceive(sim)

In [ ]:
# Its whole brain, one connection per line.
print(org.brain.describe())
print()
print('senses it actually consults:',
      [capability_utils.SENSOR_NAMES[i] for i in org.brain.needed_sensors])

In [ ]:
# What its action neurons want to do this timestep.
dict(zip(capability_utils.ACTION_NAMES, org.brain.think(org, sim)))

## Watching a population evolve

In [ ]:
sim = world(n_organisms=250, survival_criterion=CRITERIA['left'])

history = []
for generation in range(40):
    survivors = sim.run_generation()
    history.append(survivors / sim.n_organisms)

plt.plot(history)
plt.xlabel('generation')
plt.ylabel('fraction surviving')
plt.ylim(0, 1);

In [ ]:
# Where an evolved generation ends up, versus where it started.
start_x, start_y = sim.positions()
sim.run_generation()
end_x, end_y = sim.positions()

figure, axes = plt.subplots(1, 2, figsize=(10, 5), sharex=True, sharey=True)
axes[0].scatter(start_x, start_y, s=6)
axes[0].set_title('start of generation')
axes[1].scatter(end_x, end_y, s=6)
axes[1].set_title('end of generation')
for ax in axes:
    ax.set_xlim(0, sim.width)
    ax.set_ylim(0, sim.height)

## Ideas to try

- A new survival criterion at the bottom of `organism.py` (the animation shades any zone automatically).
- A new sense or action in `capability_utils.py` — evolution picks it up on the next run with no other changes.
- Turn `settings.point_mutation_rate` up and down and watch how fast the population adapts versus how stable it stays.